Historical study notebook. Its original input is unavailable in this public edition, so this is not a runnable public demo. It may require an input with the original structure and dependencies. This work used supplied learning resources and starter material; retained attribution appears with the notebook content.



In [ ]:
!apt install -y tesseract-ocr -q
!pip install pymupdf pytesseract opencv-python pillow numpy -q

import cv2, pytesseract, fitz, re, json, io
import numpy as np
from PIL import Image


In [ ]:
from google.colab import files
uploaded = files.upload()      # pick your-document.pdf


In [ ]:
doc = fitz.open("your-document.pdf")
page = doc[0]

# render at 300 DPI for sharper text (better OCR than default)
pix = page.get_pixmap(dpi=300)
img = np.array(Image.open(io.BytesIO(pix.tobytes("png"))))

gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)                                    # 1. grayscale
gray = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C,             # 2. contrast
                             cv2.THRESH_BINARY, 11, 2)
gray = cv2.bilateralFilter(gray, 9, 75, 75)                                     # 3. denoise, keep edges

display(Image.fromarray(gray))   # look at the result



In [ ]:
custom_config = r'--oem 3 -l eng'
ocr_text = pytesseract.image_to_string(gray, config=custom_config)

# clean: collapse whitespace + fix common OCR misreads
ocr_text = " ".join(ocr_text.split())
ocr_text = re.sub(r'\bL0T\b', 'LOT', ocr_text, flags=re.IGNORECASE)
ocr_text = re.sub(r'\bC0A\b', 'COA', ocr_text, flags=re.IGNORECASE)

print(ocr_text)


In [ ]:
ocr_data = pytesseract.image_to_data(gray, output_type=pytesseract.Output.DICT)

key_fields = ["VENDOR", "ANIMAL", "ORIGIN", "STATEMENT", "PRODUCT",
              "AKTA", "DATE", "SIGNATURE", "DOC1098513", "MARCH"]

extracted_data = {}
for i in range(len(ocr_data["text"])):
    word = ocr_data["text"][i].strip().upper()
    conf = int(ocr_data["conf"][i])
    x, y, w, h = ocr_data["left"][i], ocr_data["top"][i], ocr_data["width"][i], ocr_data["height"][i]
    if word and conf > 40 and any(k in word for k in key_fields):
        extracted_data[word] = {"text": word, "bounding_box": [x, y, w, h], "conf": conf}

print(json.dumps(extracted_data, indent=4))


In [ ]:
with open("ocr_extracted.json", "w") as f:
    json.dump(extracted_data, f, indent=4)
files.download("ocr_extracted.json")
